# ML Zoomcamp 2026 — Homework 1

Official: `materials/homework.md` (do not edit).
Data: `data/car_fuel_efficiency_2026.csv` (2026 pinned release).
Write-up: `ML_01_HW.md`.

Kernel: **ML Zoomcamp** / this folder `.venv`. Do not use `!pip`.


## 0) Imports


In [27]:
import os
os.environ.setdefault(r"MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault(r"XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault(r"TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault(r"TMP", r"E:\IT_SPACES\AI\.cache\tmp")

import numpy as np
import pandas as pd


## Q1. Pandas version


In [28]:
pd.__version__


'3.0.5'

## Getting the data

CSV should already be under `data/`. If missing, download the 2026 pinned file (not the old alexeygrigorev/datasets URL).


In [29]:
from pathlib import Path

DATA = Path("data") / "car_fuel_efficiency_2026.csv"
df = pd.read_csv(DATA)
df.head()


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [30]:
df.columns.tolist()


['model_year',
 'origin',
 'fuel_type',
 'drivetrain',
 'num_doors',
 'engine_displacement',
 'num_cylinders',
 'horsepower',
 'vehicle_weight',
 'acceleration',
 'fuel_efficiency_mpg']

## Q2. Records count


In [31]:
# TODO: number of records
len(df)

10000

In [32]:
df.shape

(10000, 11)

## Q3. Fuel types


In [33]:
# TODO: how many fuel types
df['fuel_type'].nunique()

3

In [34]:
df['fuel_type'].unique()

<StringArray>
['Gasoline', 'Diesel', 'Hybrid']
Length: 3, dtype: str

## Q4. Missing values


In [35]:
# TODO: how many columns have missing values
(df.isnull().sum() > 0).sum()

np.int64(2)

## Q5. Max fuel efficiency (Asia)


In [36]:
# TODO: max fuel_efficiency_mpg where origin == Asia
df[df["origin"] == 'Asia']['fuel_efficiency_mpg'].max()

np.float64(41.2)

## Q6. Median horsepower

Median → mode → fillna with mode → median again. Did it change?


In [37]:
# TODO: horsepower median / mode / fillna / median again
median_hp = df['horsepower'].median()
most_freq_hp = df['horsepower'].mode()[0]
df['horsepower'] = df['horsepower'].fillna(most_freq_hp)
new_median_hp = df['horsepower'].median()
print("Original median:", median_hp)
print("New median:",new_median_hp)
status = 'decreased' if (median_hp > new_median_hp) else ('the same' if(median_hp==new_median_hp) else 'increased')
print(f"Thus, it is {status}")

Original median: 254.0
New median: 252.0
Thus, it is decreased


## Q7. Sum of weights

공식 단계 (셀에 1~9 번호 주석):

1. Asia만 필터  
2. `vehicle_weight`, `model_year`만 선택  
3. 처음 7행  
4. NumPy 배열 `X`  
5. `XTX = X.T @ X`  
6. `XTX` 역행렬  
7. `y` 배열 생성  
8. `w = inv(XTX) @ X.T @ y`  
9. `w` 원소 합


In [38]:
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [39]:
import numpy as np

# Q7-1. origin이 'Asia'인 행만 남깁니다. (Europe은 포함하지 않습니다.)
# 왜: 숙제가 요구하는 분석 대상이 Asia 차량뿐이기 때문입니다.
df_asia = df[df['origin'] == 'Asia']

# Q7-2. vehicle_weight와 model_year 두 컬럼만 선택합니다.
# 왜: 이 두 값이 행렬 X의 열이 됩니다. (컬럼명 weight가 아니라 vehicle_weight)
df_X = df_asia[['vehicle_weight', 'model_year']]

# Q7-3. 그중 처음 7개 행만 가져옵니다.
# 왜: 숙제가 "first 7 values"로 크기를 고정했습니다. 정렬하지 말고 필터 결과 순서 그대로.
df_X7 = df_X.head(7)

# Q7-4. 판다스 표를 넘파이 배열 X로 바꿉니다. 모양은 (7, 2)여야 합니다.
# 왜: 이후 전치·행렬곱·역행렬은 넘파이 배열 위에서 계산합니다.
X = df_X7.values
print("X shape:", X.shape)  # (7, 2)인지 확인


X shape: (7, 2)


In [40]:
# Q7-5. X의 전치와 X를 곱해 XTX를 만듭니다. 모양은 (2, 2)여야 합니다.
# 왜: 선형 회귀 정규방정식에서 XᵀX 항이 필요합니다. 요소별 곱(*)이 아니라 행렬곱(@).
XTX = X.T @ X
print("XTX shape:", XTX.shape)  # (2, 2)인지 확인


XTX shape: (2, 2)


In [41]:
# Q7-6. XTX의 역행렬을 구합니다.
# 왜: 정규방정식에서 (XᵀX)⁻¹ 항이 필요합니다.
XTX_inv = np.linalg.inv(XTX)


In [42]:
# Q7-7. 타깃 벡터 y를 숙제에 주어진 값으로 만듭니다. (데이터셋에서 뽑는 값이 아닙니다.)
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])


In [43]:
# Q7-8. w = 역행렬 @ X의전치 @ y 를 계산합니다.
# 왜: 절편 없이 선형 회귀 가중치를 닫힌 해로 구하는 공식입니다.
w = XTX_inv @ X.T @ y
print("w:", w)


w: [0.13644777 0.2327492 ]


In [44]:
# Q7-9. w의 모든 원소를 더합니다. 이 숫자가 Q7 답입니다.
w_sum = w.sum()
print("sum of w:", w_sum)


sum of w: 0.36919696904925486
